# Alineación Pythia ↔ humano (AoA)

Join entre `step_stabilize` por paradigma (definido en `docs/04_experimental_design.md`) y la tabla `docs/05_human_alignment.md` / `data/human_milestones.csv`. Spearman + bootstrap.

In [1]:
from pathlib import Path

import pandas as pd

from ontogenia.human_alignment import run_human_alignment

ROOT = Path("..")
PARQUET = ROOT / "results" / "aggregated_metrics.parquet"
AOA = ROOT / "data" / "human_milestones.csv"
PARQUET.exists(), AOA.exists()

(True, True)

In [2]:
if not PARQUET.exists() or not AOA.exists():
    raise FileNotFoundError(
        "Falta parquet o CSV AoA. Requisitos:\n"
        "- python -m ontogenia aggregate --output-parquet results/aggregated_metrics.parquet\n"
        "- data/human_milestones.csv con columnas: task, aoa_months"
    )

result = run_human_alignment(
    metrics_parquet=PARQUET,
    aoa_csv=AOA,
    metric_col="acc,none",
    target_model_size="160m",  # cambiar a 14m / 410m según análisis
)

result.overlap[["task", "training_step_stabilize", "aoa_months"]].head(), result.stats

(                             task  training_step_stabilize  aoa_months
 0            blimp_adjunct_island                   4000.0          72
 1  blimp_anaphor_gender_agreement                   4000.0          47
 2  blimp_anaphor_number_agreement                   1000.0          42
 3   blimp_animate_subject_passive                   2000.0          54
 4     blimp_animate_subject_trans                    512.0          30,
         rho  p_value     ci_lo     ci_hi  n_overlap model_size metric_col
 0 -0.180298  0.16804 -0.444385  0.109861         60       160m   acc,none)

In [3]:
# Guardar outputs para paper / tracking
stats_path = ROOT / "results" / "human_alignment_stats.json"
overlap_path = ROOT / "results" / "human_alignment_overlap.parquet"

result.overlap.to_parquet(overlap_path, index=False)
stats_path.write_text(result.stats.to_json(orient="records", indent=2), encoding="utf-8")

stats_path, overlap_path

(PosixPath('../results/human_alignment_stats.json'),
 PosixPath('../results/human_alignment_overlap.parquet'))

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

overlap_plot = result.overlap.dropna(subset=["training_step_stabilize"])
x = overlap_plot["aoa_months"].values
y = overlap_plot["training_step_stabilize"].values + 1
rho, pval = spearmanr(x, y)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x, y, alpha=0.7, s=40, edgecolors="white", linewidths=0.4)
ax.set_yscale("log")
ax.set_xlabel("AoA humano (meses)")
ax.set_ylabel("Step estabilización Pythia-160m (log)")
ax.set_title(f"Spearman \u03c1 = {rho:.3f}, p = {pval:.3f} (n={len(overlap_plot)})")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../figures/fig2_inline_160m.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"\u03c1 = {rho:.3f}, p = {pval:.3f}")